# Build the portal `Transcript` table

Joins the IsoAnnot domain annotations (`merged_annotations_all_git.csv`, from
`Isoannot_merge_domain_annotations_gitv_depmapomics.ipynb`) against:

- the Ensembl 104 GTF (transcript names)
- `OmicsLongReadGTF` from Taiga (the transcripts actually expressed in the release)
- the HGNC complete set (gene symbols / DepMap gene names)
- UniProt `uniprot_sprot_varsplic` (protein isoform IDs)

Run this **locally**, not on the IsoAnnot VM. See `README.md`.

Output: `Transcript_forportal_ptc.csv`, uploaded to Taiga in the last cell.

## Configuration

Update the Taiga dataset/version pins each quarter.

### Download the Ensembl 104 GTF

The reference GTF is not committed to this repo. Download and decompress it once:

```
curl -O https://ftp.ensembl.org/pub/release-104/gtf/homo_sapiens/Homo_sapiens.GRCh38.104.chr.gtf.gz
gunzip Homo_sapiens.GRCh38.104.chr.gtf.gz
```

Then point `ENSEMBL_104_GTF` below at the result. This must be the **same Ensembl release** the
annotated IsoAnnot run used (release 104, per `Isoannot_config_annotated_transcripts.yaml`).

In [ ]:
from pathlib import Path

# Output of Isoannot_merge_domain_annotations_gitv.ipynb; copy it down off the IsoAnnot VM.
MERGED_ANNOTATIONS_CSV = Path("merged_annotations_all_git.csv")

# Ensembl 104 transcript GTF -- see the download command above.
ENSEMBL_104_GTF = Path("Homo_sapiens.GRCh38.104.chr.gtf")

# UniProt Swiss-Prot varsplic FASTA. Download once with:
#   curl -O https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot_varsplic.fasta.gz
#   gunzip uniprot_sprot_varsplic.fasta.gz
UNIPROT_VARSPLIC_FASTA = Path("uniprot_sprot_varsplic.fasta")

# Taiga pins -- bump these each quarter.
RELEASE_PERMANAME = "internal-26q3-2a74"
RELEASE_VERSION = 61
HGNC_NAME = "hgnc-gene-table-e250"
HGNC_VERSION = 5

OUTPUT_CSV = "Transcript_forportal_ptc.csv"

for p in [MERGED_ANNOTATIONS_CSV, ENSEMBL_104_GTF, UNIPROT_VARSPLIC_FASTA]:
    assert p.exists(), f"missing required input: {p}"
print("all required local inputs found")

In [ ]:
import re

import numpy as np
import pandas as pd
from Bio import SeqIO
from taigapy import TaigaClient

tc = TaigaClient()

In [ ]:
merged_annotations_all = pd.read_csv(MERGED_ANNOTATIONS_CSV)

In [ ]:
merged_annotations_all['sequence'] = merged_annotations_all['sequence'].str.replace('*', '', regex=False)

## Ensembl 104 transcript names

In [ ]:
hg38_104 = pd.read_csv(ENSEMBL_104_GTF, sep="\t", comment="#", header=None)
hg38_104.columns = [
    "seqname", "source", "feature", "start", "end", "score", "strand", "frame", "attribute",
]
hg38_104 = hg38_104[hg38_104["feature"] == "transcript"]

attr = hg38_104["attribute"]
hg38_104 = pd.DataFrame({
    "transcript_id": attr.str.extract(r'transcript_id "([^"]+)"')[0],
    "gene_id": attr.str.extract(r'gene_id "([^"]+)"')[0],
    "gene_name": attr.str.extract(r'gene_name "([^"]+)"')[0],
    "transcript_name": attr.str.extract(r'transcript_name "([^"]+)"')[0],
})
hg38_104["unique_id"] = hg38_104["transcript_id"].str.split(".").str[0]
print(hg38_104.head())

## Expressed transcripts from the release GTF

In [ ]:
OmicsLongReadGTF = tc.download_to_cache(
    name=RELEASE_PERMANAME, version=RELEASE_VERSION, file="OmicsLongReadGTF"
)

OmicsLongReadGTF_df = pd.read_parquet(OmicsLongReadGTF)
OmicsLongReadGTF_df = OmicsLongReadGTF_df[OmicsLongReadGTF_df["feature"] == "transcript"]

attr = OmicsLongReadGTF_df["attribute"]
lr_gtf = pd.DataFrame({
    "transcript_id": attr.str.extract(r'transcript_id "([^"]+)"')[0],
    "gene_id": attr.str.extract(r'gene_id "([^"]+)"')[0],
    "gene_name": attr.str.extract(r'gene_name "([^"]+)"')[0],
    "transcript_name": attr.str.extract(r'transcript_name "([^"]+)"')[0],
})
lr_gtf["unique_id"] = lr_gtf["transcript_id"].str.split(".").str[0]
print(lr_gtf.head())

In [ ]:
merged_gtf = lr_gtf.merge(hg38_104, on="unique_id", how="outer", indicator=True)
merged_gtf = merged_gtf.merge(
    merged_annotations_all,
    left_on="unique_id",
    right_on="isoform",
    how="outer",
    indicator="annotation_merge",
)

In [ ]:
# Drop transcripts present only in the annotation set, and those only in Ensembl 104
# (i.e. keep transcripts that are actually expressed in this release).
merged_gtf = merged_gtf[
    (merged_gtf["annotation_merge"] != "right_only") & (merged_gtf["_merge"] != "right_only")
]

In [ ]:
cols = [
    "transcript_id_x", "gene_id_x", "gene_name_x", "transcript_name_x", "unique_id",
] + merged_annotations_all.columns.tolist()
merged_gtf = merged_gtf[cols]

# Strip the _x suffix left by the merge (suffix only, not any interior _x).
merged_gtf.columns = [re.sub(r"_x$", "", col) for col in merged_gtf.columns]

In [ ]:
# Transcripts with no SQANTI record are non-coding.
merged_gtf["structural_category"] = merged_gtf["structural_category"].fillna("non_coding")
merged_gtf["coding"] = merged_gtf["coding"].fillna("non_coding")

In [ ]:
merged_gtf = merged_gtf[[
    "unique_id", "transcript_id", "gene_id", "gene_name", "transcript_name",
    "domain", "sequence", "structural_category", "coding", "NMD_flag",
]]

# Column set carried through the HGNC join below; captured before the join adds columns.
BASE_COLS = merged_gtf.columns.tolist()

## Merge with the HGNC table

Genes whose Ensembl ID maps to more than one HGNC symbol are ambiguous and are dropped from the
symbol lookup. Transcripts that fail the Ensembl-ID join are retried on `gene_name`.

In [ ]:
hgnc_complete_set = tc.get(name=HGNC_NAME, version=HGNC_VERSION, file="hgnc_complete_set")

hgnc_complete_set["depmap_name"] = (
    hgnc_complete_set["symbol"]
    + " ("
    + hgnc_complete_set["entrez_id"].astype("Int64").astype(str)
    + ")"
)
hgnc_complete_set = hgnc_complete_set[
    ["ensembl_gene_id", "depmap_name", "symbol"]
].drop_duplicates()

In [ ]:
# Drop Ensembl gene IDs that map to more than one symbol.
ambiguous = hgnc_complete_set.groupby("ensembl_gene_id").filter(
    lambda x: x["symbol"].nunique() > 1
)
hgnc_complete_set = hgnc_complete_set[
    ~hgnc_complete_set["ensembl_gene_id"].isin(ambiguous["ensembl_gene_id"])
]
print(f"dropped {ambiguous['ensembl_gene_id'].nunique()} ambiguous ensembl gene ids")

In [ ]:
merged_gtf["ensembl_gene_id_stripped"] = merged_gtf["gene_id"].str.split(".").str[0]

merged_gtf_1 = merged_gtf.merge(
    hgnc_complete_set,
    left_on="ensembl_gene_id_stripped",
    right_on="ensembl_gene_id",
    how="left",
)

In [ ]:
# Rows that failed the ensembl_gene_id join: retry the lookup on gene symbol.
no_hgnc = merged_gtf_1[merged_gtf_1["depmap_name"].isna()][BASE_COLS]
no_hgnc = no_hgnc.merge(hgnc_complete_set, left_on="gene_name", right_on="symbol", how="left")
no_hgnc = no_hgnc.dropna(subset=["symbol"])

merged_gtf_1 = merged_gtf_1.dropna(subset=["symbol"])
print(f"{len(no_hgnc)} rows recovered via gene_name lookup")

In [ ]:
# The two sets must be disjoint, otherwise the concat below double-counts transcripts.
overlap = merged_gtf_1[merged_gtf_1["transcript_id"].isin(no_hgnc["transcript_id"])]
assert len(overlap) == 0, f"{len(overlap)} transcripts appear in both merged_gtf_1 and no_hgnc"

merged_gtf_1 = pd.concat([merged_gtf_1, no_hgnc], ignore_index=True)

In [ ]:
# Prefer the HGNC symbol when it disagrees with the GTF gene name; fall back to
# symbol-unique_id when the transcript has no name.
merged_gtf_1["Transcript"] = np.where(
    (merged_gtf_1["symbol"] != merged_gtf_1["gene_name"])
    & (merged_gtf_1["transcript_name"].notna()),
    merged_gtf_1["symbol"]
    + "-"
    + merged_gtf_1["transcript_name"].str.split("-").str[1:].str.join("-"),
    np.where(
        merged_gtf_1["transcript_name"].isna(),
        merged_gtf_1["symbol"] + "-" + merged_gtf_1["unique_id"],
        merged_gtf_1["transcript_name"],
    ),
)

## Merge in UniProt protein isoform IDs

Joined on exact peptide sequence + gene symbol.

In [ ]:
uniprot_records = list(SeqIO.parse(str(UNIPROT_VARSPLIC_FASTA), "fasta"))

uniprot_df = pd.DataFrame([
    {"id": r.id, "description": r.description, "sequence": str(r.seq)} for r in uniprot_records
])

uniprot_df["protein_isoform_id"] = uniprot_df["id"].str.split("|").str[1]
uniprot_df["UniprotID"] = uniprot_df["protein_isoform_id"].str.split("-").str[0]
uniprot_df["species"] = (
    uniprot_df["description"].str.split("OS=").str[1].str.split(" OX=").str[0]
)
uniprot_df["gene_name"] = uniprot_df["description"].str.split("GN=").str[1]

uniprot_df = uniprot_df[uniprot_df["species"] == "Homo sapiens"]
uniprot_df = uniprot_df[["sequence", "protein_isoform_id", "UniprotID", "gene_name"]]

In [ ]:
merged_gtf_1 = merged_gtf_1.merge(
    uniprot_df,
    left_on=["sequence", "symbol"],
    right_on=["sequence", "gene_name"],
    how="left",
)

## Final column selection

In [ ]:
merged_gtf_2 = merged_gtf_1[[
    "transcript_id", "unique_id", "domain", "depmap_name", "symbol", "Transcript",
    "protein_isoform_id", "UniprotID", "structural_category", "coding", "NMD_flag",
]]
merged_gtf_2.columns = [
    "ID", "Stripped_ID", "Domain", "DepmapGeneName", "Symbol", "Transcript",
    "ProteinIsoformID", "UniprotID", "StructuralCategory", "Coding", "NMDFlag",
]

# Ensembl transcripts are "Annotated"; novel ones keep their SQANTI structural category.
merged_gtf_2["Category"] = np.where(
    merged_gtf_2["Stripped_ID"].str.startswith("ENST"),
    "Annotated",
    merged_gtf_2["StructuralCategory"],
)

merged_gtf_2 = merged_gtf_2[[
    "ID", "Category", "Coding", "NMDFlag", "Domain", "DepmapGeneName", "Symbol",
    "Transcript", "ProteinIsoformID", "UniprotID",
]]
print(f"{len(merged_gtf_2)} rows")
merged_gtf_2.head()

In [ ]:
merged_gtf_2.to_csv(OUTPUT_CSV, index=False)
print(f"wrote {OUTPUT_CSV}")

## Upload to Taiga

In [ ]:
from taigapy import create_taiga_client_v3
from taigapy.client_v3 import LocalFormat, UploadedFile

tc_v3 = create_taiga_client_v3()

updated_dataset = tc_v3.update_dataset(
    permaname=RELEASE_PERMANAME,
    reason="update Transcript.csv (hgnc geneset + novels)",
    additions=[
        UploadedFile(
            name="Transcript",
            local_path=OUTPUT_CSV,
            format=LocalFormat.CSV_TABLE,
        )
    ],
)